In [ ]:
import sys
import os
from pathlib import Path
sys.path.append(os.path.abspath("../.."))
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from tinyconformal.regressor import ConformalizedRegressor
from sklearn.datasets import fetch_california_housing
import numpy as np

In [ ]:
EXAMPLES_DIR = Path.cwd().parent
if str(EXAMPLES_DIR) not in sys.path:
    sys.path.insert(0, str(EXAMPLES_DIR))
from utils.plot_utils import plot_prediction_intervals

In [ ]:
data = fetch_california_housing(as_frame=True)
X, y = data.data, data.target

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_calib, y_train, y_calib = train_test_split(X_train, y_train, test_size=0.25, random_state=42)

In [ ]:
rf = RandomForestRegressor(random_state=42, n_jobs=-1, oob_score=True, max_depth=int(np.ceil(np.log2(len(X_train)) - 1)))
rf.fit(X_train, y_train)

In [ ]:
reg = ConformalizedRegressor(rf, alpha=0.20)
reg.fit(X=X_calib, y=y_calib)

In [ ]:
y_pred_intervals = reg.predict_interval(X_test)
y_pred = reg.predict(X_test)

In [ ]:
reg.evaluate(X_test, y_test)

In [ ]:
plot_prediction_intervals(y_pred_intervals[:10], y_pred[:10], y_test[:10], fig_type="png")